# U23 — Cloud Storage & Databases (Part 1): Lab

### Real-world brief: a storage layer for an engineering asset & inspection platform

A maintenance team has an **asset register** (pumps, motors, valves…) and a long **inspection history**. They need to store reports as files, query records transactionally, model the data two ways (relational vs document), and understand how it would **shard** across machines as it grows.

You'll work hands-on with the storage ideas from U23 using only the Python standard library (`sqlite3`, filesystem, JSON) — no cloud account required, but every concept maps directly to S3, RDS, DynamoDB and friends:
- **Object storage** — a local bucket with keys, metadata and tiering
- **Relational (SQLite)** — schema, ACID transactions, indexing
- **NoSQL document modelling** — the same data as JSON documents
- **Sharding & replication** — routing keys across shards; replica lag

**Resources provided:** `assets.csv` (200 assets) and `inspections.csv` (~4.9k inspections).

_Phase F — Data Engineering._

#objectives

Build an object store with keys, metadata and storage tiers

Model relational tables and run ACID transactions (commit/rollback)

Measure how an index speeds queries; read a query plan

Model the same data as a NoSQL document and compare trade-offs

Shard data by key across nodes and reason about skew & replication

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [1]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def build_assets(assets_path="assets.csv", insp_path="inspections.csv", seed=223, verbose=False):
    """Engineering asset register + inspection history for the storage/database lab (U23).

      assets.csv       one row per physical asset (master data)
      inspections.csv  many inspection events per asset (transactional data)

    Two related tables — perfect for relational modelling (joins, indexing, transactions),
    document modelling (asset + its inspections as one JSON doc) and sharding by asset key.
    """
    rng = np.random.default_rng(seed)

    NA = 200
    types = rng.choice(["pump", "valve", "motor", "compressor", "heat_exchanger"], NA,
                        p=[0.3, 0.25, 0.2, 0.15, 0.1])
    sites = rng.choice(["Plant_A", "Plant_B", "Plant_C"], NA, p=[0.45, 0.35, 0.2])
    assets = pd.DataFrame({
        "asset_id": [f"A{i:04d}" for i in range(1, NA + 1)],
        "asset_type": types, "site": sites,
        "install_date": pd.to_datetime("2015-01-01") + pd.to_timedelta(rng.integers(0, 3300, NA), unit="D"),
        "criticality": rng.choice(["low", "medium", "high"], NA, p=[0.4, 0.4, 0.2]),
        "manufacturer": rng.choice(["Acme", "Bharat", "Crompton", "Danfoss"], NA),
    })

    # inspections: variable count per asset
    rows = []
    iid = 1
    inspectors = ["E. Rao", "S. Khan", "M. Iyer", "P. Singh", "L. Dsouza"]
    for _, a in assets.iterrows():
        k = rng.integers(5, 45)                       # number of inspections
        dates = pd.to_datetime("2022-01-01") + pd.to_timedelta(np.sort(rng.integers(0, 850, k)), unit="D")
        crit_w = {"low": 0.05, "medium": 0.12, "high": 0.22}[a.criticality]
        for d in dates:
            fail = rng.random() < crit_w
            result = "fail" if fail else ("repair" if rng.random() < 0.15 else "pass")
            rows.append({
                "inspection_id": f"I{iid:06d}", "asset_id": a.asset_id, "inspection_date": d,
                "inspector": rng.choice(inspectors),
                "result": result,
                "health_score": round(float(np.clip(rng.normal(85 - (25 if fail else 0), 8), 0, 100)), 1),
                "duration_min": int(np.clip(rng.normal(45, 15), 5, None)),
                "cost_inr": int(np.clip(rng.gamma(2.0, 1800), 200, None)),
            })
            iid += 1
    insp = pd.DataFrame(rows)

    assets.to_csv(assets_path, index=False)
    insp.to_csv(insp_path, index=False)
    if verbose:
        print("assets:", assets.shape, "| inspections:", insp.shape)
        print("inspections per asset: min/median/max =",
              insp.groupby("asset_id").size().agg(["min", "median", "max"]).to_dict())
        print("result mix:", insp.result.value_counts().to_dict())
        print("site distribution:", assets.site.value_counts().to_dict())
    return assets, insp

if not (os.path.exists('assets.csv') and os.path.exists('inspections.csv')):
    build_assets(); print('Generated source files.')
else:
    print('Found the provided source files.')

Generated source files.


In [2]:
import pandas as pd, numpy as np, sqlite3, json, time, os, shutil
assets = pd.read_csv('assets.csv', parse_dates=['install_date'])
insp = pd.read_csv('inspections.csv', parse_dates=['inspection_date'])
print('assets:', assets.shape, '| inspections:', insp.shape)
assets.head(3)

assets: (200, 6) | inspections: (4876, 8)


,asset_id,asset_type,site,install_date,criticality,manufacturer
0,A0001,compressor,Plant_A,2023-06-01,medium,Crompton
1,A0002,valve,Plant_B,2019-07-02,low,Bharat
2,A0003,motor,Plant_B,2015-09-10,high,Crompton


# Part A — Object storage
Object storage (think S3) keeps each file as an **object** — bytes plus metadata — under a unique **key**, in a flat namespace called a **bucket**. We'll simulate one with a directory.

In [3]:
# -----------------------------------------------------------
# 🔹 A1. A LOCAL OBJECT STORE: put / get / list / tier
# -----------------------------------------------------------
class ObjectStore:
    def __init__(self, bucket='bucket'):
        self.root = bucket; shutil.rmtree(bucket, ignore_errors=True); os.makedirs(bucket)
        self.meta = {}        # key -> metadata dict (tier, size, content_type)
    def put(self, key, data, tier='hot', content_type='application/json'):
        path = os.path.join(self.root, key.replace('/', '__'))
        blob = json.dumps(data).encode()
        with open(path, 'wb') as f: f.write(blob)
        self.meta[key] = {'tier': tier, 'size': len(blob), 'content_type': content_type}
    def get(self, key):
        path = os.path.join(self.root, key.replace('/', '__'))
        with open(path, 'rb') as f: return json.loads(f.read())
    def list(self, prefix=''):
        return sorted(k for k in self.meta if k.startswith(prefix))
    def set_tier(self, key, tier):
        self.meta[key]['tier'] = tier

store = ObjectStore('inspection_bucket')
print('empty bucket created.')

empty bucket created.


In [4]:
# -----------------------------------------------------------
# 🔹 A2. STORE inspection reports as objects, keyed by asset & date
# -----------------------------------------------------------
for r in insp.itertuples(index=False):
    key = f'inspections/{r.asset_id}/{r.inspection_date.date()}_{r.inspection_id}.json'
    store.put(key, {'inspector': r.inspector, 'result': r.result,
                    'health_score': r.health_score, 'cost_inr': int(r.cost_inr)})
print('objects stored:', len(store.meta))
# flat namespace, but prefixes act like folders:
a1_keys = store.list('inspections/A0001/')
print('objects under asset A0001:', len(a1_keys))
print('example key:', a1_keys[0])
print('fetch one object:', store.get(a1_keys[0]))

objects stored: 4876
objects under asset A0001: 12
example key: inspections/A0001/2022-02-27_I000001.json
fetch one object: {'inspector': 'S. Khan', 'result': 'pass', 'health_score': 87.9, 'cost_inr': 1458}


In [18]:
import datetime

# 1-2. archive old objects; report counts & bytes per tier
archive_date_limit = datetime.date(2022, 7, 1)

for key, meta in store.meta.items():
    # Example key: 'inspections/A0001/2022-02-27_I000001.json'
    # Extract the date part: '2022-02-27'
    date_str = key.split('/')[2].split('_')[0]
    obj_date = datetime.datetime.strptime(date_str, '%Y-%m-%d').date()

    if obj_date < archive_date_limit:
        store.set_tier(key, 'archive')

hot_count = 0
archive_count = 0
hot_bytes = 0
archive_bytes = 0

for key, meta in store.meta.items():
    if meta['tier'] == 'hot':
        hot_count += 1
        hot_bytes += meta['size']
    elif meta['tier'] == 'archive':
        archive_count += 1
        archive_bytes += meta['size']

print(f"Objects in 'hot' tier: {hot_count} ({hot_bytes} bytes)")
print(f"Objects in 'archive' tier: {archive_count} ({archive_bytes} bytes)")

# 3. what a lifecycle policy automates: ...   (comment)
# A real lifecycle policy automates the transition of objects between storage tiers
# based on predefined rules (e.g., age of object, last access time).
# Retrieval from archive is typically slower and cheaper because archived data is
# moved to less accessible storage mediums (like magnetic tape) or compressed heavily.
# This reduces operational costs for data that is infrequently accessed, but incurs a
# latency cost when that data needs to be retrieved, as it must be moved back to
# 'hotter' storage or decompressed.

Objects in 'hot' tier: 3828 (316380 bytes)
Objects in 'archive' tier: 1048 (86535 bytes)


# Part B — Relational database (SQLite)
A relational store keeps data in typed tables linked by keys, with **ACID transactions** and fast **indexed** queries. SQLite gives us all of that in a single file.

In [19]:
# -----------------------------------------------------------
# 🔹 B1. CREATE schema with a primary key and a foreign key
# -----------------------------------------------------------
DB = 'assets.db'
if os.path.exists(DB): os.remove(DB)
con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')
con.execute('''CREATE TABLE assets (
    asset_id TEXT PRIMARY KEY, asset_type TEXT, site TEXT,
    install_date TEXT, criticality TEXT, manufacturer TEXT)''')
con.execute('''CREATE TABLE inspections (
    inspection_id TEXT PRIMARY KEY, asset_id TEXT, inspection_date TEXT,
    inspector TEXT, result TEXT, health_score REAL, duration_min INTEGER, cost_inr INTEGER,
    FOREIGN KEY (asset_id) REFERENCES assets(asset_id))''')
assets.astype(str).to_sql('assets', con, if_exists='append', index=False)
insp.astype({'inspection_date': str}).to_sql('inspections', con, if_exists='append', index=False)
con.commit()
print('loaded:', con.execute('SELECT COUNT(*) FROM assets').fetchone()[0], 'assets,',
      con.execute('SELECT COUNT(*) FROM inspections').fetchone()[0], 'inspections')

loaded: 200 assets, 4876 inspections


In [20]:
# -----------------------------------------------------------
# 🔹 B2. ACID TRANSACTION — all-or-nothing writes
# Record an inspection AND bump the asset's manufacturer note together; if anything
# fails mid-way, the whole transaction rolls back, leaving the DB untouched.
# -----------------------------------------------------------
def add_inspection_txn(con, asset_id, inspection_id, score, force_fail=False):
    try:
        con.execute('BEGIN')
        con.execute('INSERT INTO inspections (inspection_id, asset_id, inspection_date, result, health_score)'
                    ' VALUES (?,?,?,?,?)', (inspection_id, asset_id, '2024-07-01', 'pass', score))
        if force_fail:
            raise ValueError('simulated failure after the first write')
        con.execute('UPDATE assets SET manufacturer = manufacturer WHERE asset_id = ?', (asset_id,))
        con.execute('COMMIT')
        return 'committed'
    except Exception as e:
        con.execute('ROLLBACK')
        return f'rolled back ({e})'

before = con.execute('SELECT COUNT(*) FROM inspections').fetchone()[0]
print('failing txn :', add_inspection_txn(con, 'A0001', 'I999001', 90, force_fail=True))
mid = con.execute('SELECT COUNT(*) FROM inspections').fetchone()[0]
print('good txn    :', add_inspection_txn(con, 'A0001', 'I999002', 90, force_fail=False))
after = con.execute('SELECT COUNT(*) FROM inspections').fetchone()[0]
print(f'counts: before={before}  after-failed-txn={mid}  after-good-txn={after}')
print('-> the failed transaction left NO partial row (atomicity).')

failing txn : rolled back (simulated failure after the first write)
good txn    : committed
counts: before=4876  after-failed-txn=4876  after-good-txn=4877
-> the failed transaction left NO partial row (atomicity).


In [21]:
# -----------------------------------------------------------
# 🔹 B3. INDEXING — make a filtered query fast
# Amplify the table so the timing difference is visible.
# -----------------------------------------------------------
big = pd.concat([insp.assign(inspection_id=insp.inspection_id + f'_{k}') for k in range(40)], ignore_index=True)
big.astype({'inspection_date': str}).to_sql('insp_big', con, if_exists='replace', index=False)
print('insp_big rows:', con.execute('SELECT COUNT(*) FROM insp_big').fetchone()[0])

def timed(sql, params=()):
    t = time.perf_counter(); con.execute(sql, params).fetchall(); return (time.perf_counter() - t) * 1000

q = 'SELECT * FROM insp_big WHERE asset_id = ?'
print('plan BEFORE index:', con.execute('EXPLAIN QUERY PLAN ' + q, ('A0123',)).fetchall()[0][-1])
t_before = np.mean([timed(q, ('A0123',)) for _ in range(5)])
con.execute('CREATE INDEX idx_assetid ON insp_big(asset_id)'); con.commit()
print('plan AFTER index :', con.execute('EXPLAIN QUERY PLAN ' + q, ('A0123',)).fetchall()[0][-1])
t_after = np.mean([timed(q, ('A0123',)) for _ in range(5)])
print(f'query time: {t_before:.2f} ms (scan)  ->  {t_after:.2f} ms (indexed)  =  {t_before/max(t_after,1e-6):.1f}x faster')

insp_big rows: 195040
plan BEFORE index: SCAN insp_big
plan AFTER index : SEARCH insp_big USING INDEX idx_assetid (asset_id=?)
query time: 18.02 ms (scan)  ->  1.41 ms (indexed)  =  12.8x faster


In [22]:
# 1. join query: avg health by site for high-criticality assets
sql_query = """
SELECT
    a.site,
    AVG(i.health_score) AS average_health_score
FROM
    assets AS a
JOIN
    inspections AS i ON a.asset_id = i.asset_id
WHERE
    a.criticality = 'high'
GROUP BY
    a.site
ORDER BY
    a.site;
"""
high_crit_avg_health = pd.read_sql(sql_query, con)
print('Average health score for high-criticality assets by site:')
display(high_crit_avg_health)

# 2. composite index + EXPLAIN QUERY PLAN
# Create a composite index on insp_big(asset_id, inspection_date)
con.execute('CREATE INDEX idx_assetid_date ON insp_big(asset_id, inspection_date)'); con.commit()

# Query to demonstrate the use of the composite index
q_composite = 'SELECT * FROM insp_big WHERE asset_id = ? AND inspection_date = ?'
print('\nplan AFTER composite index:', con.execute('EXPLAIN QUERY PLAN ' + q_composite, ('A0123', '2022-03-01')).fetchall()[0][-1])

# 3. cost of indexes: ...   (comment)
# Cost of indexes:
# - Slower writes: Every INSERT, UPDATE, or DELETE operation on an indexed table
#   requires updating the index structure, which adds overhead and slows down write operations.
# - Extra storage: Indexes consume disk space, which can be significant for large tables
#   or tables with many indexes. This adds to storage costs.
# - Increased complexity: Managing indexes adds complexity to database administration,
#   requiring decisions on which columns to index and when to rebuild indexes.

Average health score for high-criticality assets by site:


,site,average_health_score
0,Plant_A,80.219824
1,Plant_B,79.974652
2,Plant_C,80.016860



plan AFTER composite index: SEARCH insp_big USING INDEX idx_assetid_date (asset_id=? AND inspection_date=?)


# Part C — NoSQL document modelling
A document store (think MongoDB/DynamoDB) keeps related data **together** as one JSON document, optimised for fetch-by-key access rather than joins.

In [24]:
# -----------------------------------------------------------
# 🔹 C1. The same data as ONE document per asset (denormalised)
# -----------------------------------------------------------
def build_asset_document(asset_id):
    a = assets[assets.asset_id == asset_id].iloc[0].to_dict()
    a['install_date'] = str(a['install_date'].date())
    rows = insp[insp.asset_id == asset_id]
    a['inspections'] = [{'date': str(r.inspection_date.date()), 'result': r.result,
                         'health_score': r.health_score} for r in rows.itertuples()]
    a['inspection_count'] = len(rows)
    return a

doc = build_asset_document('A0001')
print('document keys:', list(doc.keys()))
print('embedded inspections:', doc['inspection_count'])
print(json.dumps({k: doc[k] for k in ['asset_id', 'asset_type', 'site', 'inspection_count']}, indent=2))
print('first embedded inspection:', doc['inspections'][0])

document keys: ['asset_id', 'asset_type', 'site', 'install_date', 'criticality', 'manufacturer', 'inspections', 'inspection_count']
embedded inspections: 12
{
  "asset_id": "A0001",
  "asset_type": "compressor",
  "site": "Plant_A",
  "inspection_count": 12
}
first embedded inspection: {'date': '2022-02-27', 'result': 'pass', 'health_score': 87.9}


In [25]:
# -----------------------------------------------------------
# 🔹 C2. Access pattern: 'show everything about asset X'
# Document = ONE read. Relational = a join across two tables.
# -----------------------------------------------------------
docs = {aid: build_asset_document(aid) for aid in assets.asset_id}   # the 'document store'
d = docs['A0050']
print('document store: 1 key lookup ->', d['asset_id'], 'with', d['inspection_count'], 'inspections')
rel = pd.read_sql('SELECT a.*, i.inspection_date, i.result FROM assets a '
                  'JOIN inspections i ON a.asset_id=i.asset_id WHERE a.asset_id="A0050"', con)
print('relational: a join returning', len(rel), 'rows that the app must regroup')

document store: 1 key lookup -> A0050 with 43 inspections
relational: a join returning 43 rows that the app must regroup


In [26]:
# 1. average embedded inspections per document
total_inspections = sum(doc['inspection_count'] for doc in docs.values())
average_fan_out = total_inspections / len(docs)
print(f"Average number of embedded inspections per document (fan-out): {average_fan_out:.2f}")

# 2. update anomaly — relational vs document: ...   (comment)
# Suppose an inspector's name changes. In a relational model, this update is easy
# because the inspector's name is stored in a single row in an 'inspectors' table
# (or similar) and referred to by foreign keys in the 'inspections' table. A single
# UPDATE statement on that one row is sufficient.
#
# In contrast, in the document model (where inspector details are embedded directly
# within each inspection or asset document), if an inspector's name changes, every
# document that contains an inspection performed by that inspector would need to be
# located and updated. This involves reading, modifying, and rewriting potentially
# many documents, which is a much more painful and resource-intensive operation.
# This illustrates the classic normalisation vs. read-speed trade-off: denormalization
# (embedding data) speeds up reads (fewer joins/lookups) but complicates writes (data duplication).

Average number of embedded inspections per document (fan-out): 24.38


# Part D — Sharding & replication
When one node can't hold the data, **shard** it: split rows across nodes by a shard key. We'll route assets across shards and watch what happens with a skewed key, then simulate a read **replica**.

In [27]:
# -----------------------------------------------------------
# 🔹 D1. SHARD assets across nodes by a hash of the key
# -----------------------------------------------------------
import zlib
N_SHARDS = 4
def shard_of(key, n=N_SHARDS):
    return zlib.crc32(str(key).encode()) % n

shards = {i: [] for i in range(N_SHARDS)}
for aid in assets.asset_id:
    shards[shard_of(aid)].append(aid)
print('assets per shard (hash key -> even spread):', {i: len(v) for i, v in shards.items()})

assets per shard (hash key -> even spread): {0: 51, 1: 50, 2: 49, 3: 50}


In [28]:
# -----------------------------------------------------------
# 🔹 D2. A BAD shard key causes a hot shard (skew)
# Sharding by 'site' (only 3 values, unevenly sized) overloads one node.
# -----------------------------------------------------------
site_shards = assets.groupby('site').size()
print('rows per shard when sharding by site:')
print(site_shards.to_string())
print('-> Plant_A is a HOT shard; the cluster is only as fast as its busiest node.')

rows per shard when sharding by site:
site
Plant_A    82
Plant_B    72
Plant_C    46
-> Plant_A is a HOT shard; the cluster is only as fast as its busiest node.


In [29]:
# -----------------------------------------------------------
# 🔹 D3. REPLICATION & eventual consistency (a tiny simulation)
# Primary takes writes; a replica lags slightly before catching up.
# -----------------------------------------------------------
primary = {'A0001_status': 'ok'}
replica = dict(primary)        # in sync to start
primary['A0001_status'] = 'fault'      # a write hits the primary
print('immediately after write:')
print('  read from primary:', primary['A0001_status'])
print('  read from replica:', replica['A0001_status'], '(stale -> eventual consistency)')
replica.update(primary)        # replication catches up
print('after replication catches up, replica:', replica['A0001_status'])

immediately after write:
  read from primary: fault
  read from replica: ok (stale -> eventual consistency)
after replication catches up, replica: fault


In [30]:
# 1. range-shard the asset_id space; show per-shard counts
all_asset_ids = sorted(assets.asset_id.unique())

# Determine the range boundaries for sharding
shard_size = len(all_asset_ids) // N_SHARDS
range_shards = {i: [] for i in range(N_SHARDS)}

for i in range(N_SHARDS):
    start_idx = i * shard_size
    end_idx = (i + 1) * shard_size if i < N_SHARDS - 1 else len(all_asset_ids)
    current_shard_ids = all_asset_ids[start_idx:end_idx]

    for asset_id in current_shard_ids:
        range_shards[i].append(asset_id)

print('assets per shard (range key):', {i: len(v) for i, v in range_shards.items()})

# 2. hash vs range sharding choice: ...   (comment)
# Compare hash vs range sharding:
# - Range sharding: Allows for efficient range scans (e.g., all assets from 'A0010' to 'A0050' are likely on the same shard).
#   However, it risks creating 'hot spots' if data with sequential keys is accessed disproportionately (e.g., new assets always have higher IDs and are added to one shard).
# - Hash sharding: Spreads data evenly across shards, reducing the chance of hot spots from sequential access.
#   But it scatters related data (e.g., 'A0010' and 'A0011' could be on different shards), making range scans inefficient as they require querying multiple shards.

# Which would you pick for time-series inspection data, and why?
# For time-series inspection data, where queries often involve date ranges (e.g., 'all inspections from last week'),
# **range sharding by `inspection_date`** would generally be preferred. This is because time-series data naturally
# has sequential keys (dates). Range sharding would ensure that inspections within a specific time window are co-located
# on the same or a small number of shards, making range queries highly efficient. Hash sharding, while distributing writes
# evenly, would scatter time-series data across all shards, making range queries very expensive as they would need to hit every shard.

assets per shard (range key): {0: 50, 1: 50, 2: 50, 3: 50}


In [31]:
con.close()
print('database connection closed.')

database connection closed.
